# Network Intrusion Detection — Progressive Dataset Evaluation

**Paper:** Chua & Salam (2023), *Evaluation of ML Algorithms in Network-Based Intrusion Detection Using Progressive Dataset*, Symmetry 15, 1251

**Setup:** Run this header cell first every time you open a new Colab session.

In [ ]:
# ── Header cell: run this first in every new Colab session ──────────────────
import sys, os

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repo so src/ modules are importable
REPO_URL = 'https://github.com/Rosette28/data-science-cyber-final-project'  # ← update
REPO_DIR = '/content/ids-project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 3. Install dependencies
!pip install -q -r {REPO_DIR}/requirements.txt

print('Environment ready.')

In [ ]:
# ── Global configuration — only line you change between runs ─────────────────
DATA_DIR = '/content/drive/MyDrive/ids_data/raw/'  # ← set to your Drive folder

SEED = 42
SUBSAMPLE_FRAC = 0.10  # 10% of each day's CSV, read at load time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump, load

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='tab10')

print(f'DATA_DIR = {DATA_DIR}')

---
## §1 — Data Loading & Initial Inspection

Load CIC-IDS2017 (train) and CSE-CIC-IDS2018 (progressive test), keeping ~10% via chunked reading. Inspect shape, dtypes, memory usage, column names, and temporal structure.

In [ ]:
from src.data_loading import load_cic2017, load_cic2018, align_schemas

df_train = load_cic2017(DATA_DIR, subsample_frac=SUBSAMPLE_FRAC, seed=SEED)
df_test  = load_cic2018(DATA_DIR, subsample_frac=SUBSAMPLE_FRAC, seed=SEED)
df_train, df_test = align_schemas(df_train, df_test)

print('Train shape:', df_train.shape)
print('Test  shape:', df_test.shape)

### Shape, dtypes, memory, and column name inspection

In [ ]:
# ── Shape and memory ──────────────────────────────────────────────────────────
for name, df in [('Train (CIC-IDS2017)', df_train), ('Test  (CSE-CIC-IDS2018)', df_test)]:
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"{name}: {df.shape[0]:>7,} rows × {df.shape[1]} cols  |  {mem_mb:.1f} MB")

print()

# ── Dtype breakdown ───────────────────────────────────────────────────────────
print("Train dtype counts:")
print(df_train.dtypes.value_counts().to_string())
print("\nTest dtype counts:")
print(df_test.dtypes.value_counts().to_string())

In [ ]:
# ── Column name analysis ──────────────────────────────────────────────────────
# Features fall into five semantic groups derived from CICFlowMeter's documentation.
feature_groups = {
    'Packet length stats':    [c for c in df_train.columns if 'Packet Length' in c or 'Pkt Len' in c or 'Packet Size' in c or 'Segment Size' in c],
    'Packet counts / rates':  [c for c in df_train.columns if 'Packets' in c or 'Pkts' in c or 'Bytes' in c or 'Bulk' in c or 'Subflow' in c],
    'Inter-arrival times':    [c for c in df_train.columns if 'IAT' in c or 'Flow Duration' in c],
    'TCP flags':              [c for c in df_train.columns if 'Flag' in c or 'Win' in c],
    'Other / port / misc':    [c for c in df_train.columns if c not in sum([
        [c for c in df_train.columns if 'Packet Length' in c or 'Pkt Len' in c or 'Packet Size' in c or 'Segment Size' in c],
        [c for c in df_train.columns if 'Packets' in c or 'Pkts' in c or 'Bytes' in c or 'Bulk' in c or 'Subflow' in c],
        [c for c in df_train.columns if 'IAT' in c or 'Flow Duration' in c],
        [c for c in df_train.columns if 'Flag' in c or 'Win' in c],
        ['Label']
    ], []) and c != 'Label'],
}

print("Feature groups (76 features total after schema alignment):\n")
for group, cols in feature_groups.items():
    print(f"  {group} ({len(cols)}): {cols}")

print(f"\nLabel column: 'Label' — unique values in train: {df_train['Label'].unique()}")
print(f"                       — unique values in test:  {df_test['Label'].unique()}")

**Shape and memory:** After loading and schema alignment, both datasets have 77 columns (76 features + Label). The training set (CIC-IDS2017, 10% sample) holds 281,434 rows at 159.7 MB; the progressive test set (CSE-CIC-IDS2018) holds 124,696 rows at 70.9 MB — comfortably within free Colab RAM.

**Dtype breakdown:** Both datasets share the same pattern: 52 `int64` columns, 24 `float32`, and 1 `object` (Label). The split makes sense — flag counts and raw packet/byte counts are inherently discrete integers and were never cast. Rates, averages, standard deviations and variances are continuous and were downcast from `float64` to `float32`, halving their memory footprint with negligible precision loss for ML.

**Feature groups (76 features across 5 semantic groups):**
- **Packet length stats (16):** Min, max, mean, std, variance of packet sizes in both forward and backward direction, plus overall flow statistics. Capture *what is being sent* — attack flows often show extremely uniform sizes (e.g. flooding with fixed-size packets) or extreme values.
- **Packet counts and byte rates (18):** Total packet/byte counts, flow-level rates (packets/s, bytes/s), subflow counts, bulk transfer stats. Capture *volume and asymmetry* — DoS floods show extreme forward rates with near-zero backward traffic; exfiltration shows the reverse.
- **Inter-arrival times (15):** Min, max, mean, std of gaps between packets within a flow, plus total flow duration. Capture *timing patterns* — port scans and flooding attacks have unusually regular or extremely small IATs compared to human-generated traffic.
- **TCP flags (14):** Counts of SYN/FIN/RST/PSH/ACK/URG/ECE/CWE flags, plus initial TCP window sizes (forward and backward). Capture *connection behaviour* — a high SYN count with no matching ACK is a textbook SYN flood; window sizes can fingerprint operating systems and reveal botnet clients.
- **Other / port / misc (13):** Destination port, header lengths, active/idle time statistics, and two CICFlowMeter-specific fields (`act_data_pkt_fwd`, `min_seg_size_forward`). Destination port alone is a strong discriminator — web traffic concentrates on 80/443, SSH on 22, scanning generates traffic across unusual high ports.

**Labels:** The training set contains 15 distinct attack types plus BENIGN. The test set contains 10 attack types plus BENIGN, with all labels now decoded to human-readable strings (2018 used integer encoding in the pre-processed file; 2017 had UTF-8 encoding artifacts in the Web Attack labels — both fixed at load time).

**No timestamp column in features:** CICFlowMeter writes a `Timestamp` field to the raw CSVs, but it is deliberately excluded from the feature matrix. Using it would constitute time leakage — a model could learn "flows from 2018 = test distribution" rather than genuine traffic patterns. The temporal structure is enforced at the dataset-split level, not the feature level.

### Temporal structure and the progressive evaluation design

In [ ]:
import os, glob

# ── Per-day file breakdown for CIC-IDS2017 ────────────────────────────────────
cic2017_dir = os.path.join(DATA_DIR, 'cic2017')
cic2018_dir = os.path.join(DATA_DIR, 'cic2018')

print("CIC-IDS2017 source files (training set):")
for f in sorted(glob.glob(os.path.join(cic2017_dir, '*.csv'))):
    print(f"  {os.path.basename(f)}")

print("\nCSE-CIC-IDS2018 source files (progressive test set):")
for f in sorted(glob.glob(os.path.join(cic2018_dir, '*.csv'))):
    print(f"  {os.path.basename(f)}")

In [ ]:
# ── Attack type coverage in each dataset ─────────────────────────────────────
print("Attack types in TRAINING set (CIC-IDS2017):")
train_labels = df_train['Label'].value_counts()
print(train_labels.to_string())

print("\nAttack types in TEST set (CSE-CIC-IDS2018):")
test_labels = df_test['Label'].value_counts()
print(test_labels.to_string())

# ── Semantic grouping — map both datasets to a shared attack family ───────────
# The two datasets use different naming conventions for the same attack families.
# We normalise to a common family name before comparing.
_FAMILY = {
    # 2017 names
    'BENIGN':                    'Benign',
    'Bot':                       'Bot',
    'DDoS':                      'DDoS',
    'PortScan':                  'PortScan',
    'FTP-Patator':               'Brute Force - FTP',
    'SSH-Patator':               'Brute Force - SSH',
    'DoS slowloris':             'DoS - Slowloris',
    'DoS Slowhttptest':          'DoS - SlowHTTPTest',
    'DoS Hulk':                  'DoS - Hulk',
    'DoS GoldenEye':             'DoS - GoldenEye',
    'Heartbleed':                'Heartbleed',
    'Infiltration':              'Infiltration',
    'Web Attack - Brute Force':  'Brute Force - Web',
    'Web Attack - XSS':          'Brute Force - XSS',
    'Web Attack - Sql Injection':'SQL Injection',
    # 2018 names
    'Brute Force - Web':         'Brute Force - Web',
    'Brute Force - XSS':         'Brute Force - XSS',
    'DDoS - HOIC':               'DDoS',
    'DDoS - LOIC-UDP':           'DDoS',
    'DDoS - LOIC-HTTP':          'DDoS',
    'DoS - GoldenEye':           'DoS - GoldenEye',
    'DoS - Hulk':                'DoS - Hulk',
    'DoS - SlowHTTPTest':        'DoS - SlowHTTPTest',
    'DoS - Slowloris':           'DoS - Slowloris',
}

train_families = set(df_train['Label'].map(_FAMILY).dropna().unique())
test_families  = set(df_test['Label'].map(_FAMILY).dropna().unique())

in_both     = train_families & test_families
only_train  = train_families - test_families
only_test   = test_families  - train_families

print(f"\nAttack families in BOTH datasets:        {sorted(in_both)}")
print(f"Attack families ONLY in train (2017):    {sorted(only_train)}")
print(f"Attack families ONLY in test  (2018):    {sorted(only_test)}")

**Temporal structure — what time means in this project:**

**1. Within each dataset (daily granularity):**
CIC-IDS2017 is split across 8 CSV files covering five working days (Monday 3 July – Friday 7 July 2017). Monday is benign-only background traffic. From Tuesday onward, increasingly complex attacks are injected: Tuesday has brute-force (FTP-Patator, SSH-Patator); Wednesday has DoS/DDoS; Thursday has Web Attacks and Infiltration; Friday has DDoS and PortScan. CSE-CIC-IDS2018 arrives as a single pre-processed file — the within-day structure has been collapsed.

**2. Between datasets — the progressive gap (~8 months):**
All 2017 data precedes all 2018 data. There is zero temporal overlap. This is the core of the paper's methodology: a model trained on July 2017 traffic is evaluated on February–March 2018 traffic it has never seen.

**Class imbalance in both datasets:**
- Train: 226,117 BENIGN / 55,317 attacks → **80.4% benign**
- Test: 96,421 BENIGN / 28,475 attacks → **77.2% benign**

Both datasets reflect realistic network conditions where benign traffic dominates. This imbalance is a central methodological issue — the authors address it by downsampling to a 1:1 ratio before training.

**Attack family coverage — the full picture:**

After normalising naming differences between the two datasets (e.g. "DoS Hulk" → "DoS - Hulk", "DDoS" / "DDoS - HOIC" / "DDoS - LOIC-*" → "DDoS"), the comparison looks very different from a naive string match:

- **In both datasets (9 families):** Benign, Bot, Brute Force - Web, Brute Force - XSS, DDoS, DoS - GoldenEye, DoS - Hulk, DoS - SlowHTTPTest, DoS - Slowloris
- **Only in train (6 families):** Brute Force - FTP, Brute Force - SSH, Heartbleed, Infiltration, PortScan, SQL Injection
- **Genuinely new in test: none.**

This is a critical finding for interpreting the paper's results. The 2018 test set does not introduce any attack family the model has never seen — every attack family in 2018 has a direct semantic equivalent in the 2017 training set. The models are not being asked to detect unknown attack types; they are being asked to detect the same families manifesting through different tools and in a changed network environment.

**What actually causes the performance drop then?**
The distribution shift between 2017 and 2018 is not about new attack types — it is about:
1. **Different tools within the same family:** 2017's generic "DDoS" was generated by specific tools; 2018 uses LOIC and HOIC, which generate different flow-level signatures (different packet rates, sizes, timing patterns) even though the attack intent is identical.
2. **Shifted class frequencies:** In 2017, "Web Attack - Brute Force" has only 150 samples (the rarest attack after Heartbleed and SQL Injection). In 2018, "Brute Force - XSS" is the *largest* attack class with 13,678 samples. A model that barely saw this pattern in training is now asked to classify it at high volume.
3. **Changed network environment:** Different machines, different topology, different background benign traffic distribution.

**This refines the concept drift argument:** The paper calls the performance drop "overfitting", implying models memorised 2017-specific noise. A more precise description is that the models learned tool-specific signatures rather than family-level traffic patterns — which looks like overfitting on training data but is really a failure to generalise within the same attack family across different implementations. We will test this directly in Phase 8.

### Data hygiene scan

In [ ]:
# ── Duplicate rows ────────────────────────────────────────────────────────────
feat_cols = [c for c in df_train.columns if c != 'Label']

train_dups = df_train.duplicated(subset=feat_cols).sum()
test_dups  = df_test.duplicated(subset=feat_cols).sum()
print(f"Duplicate feature rows — train: {train_dups:,}  |  test: {test_dups:,}")

# ── Remaining NaN / inf (should be zero after _clean) ────────────────────────
train_nan = df_train[feat_cols].isnull().sum().sum()
test_nan  = df_test[feat_cols].isnull().sum().sum()
print(f"Remaining NaN values  — train: {train_nan}  |  test: {test_nan}")

train_inf = np.isinf(df_train[feat_cols].values).sum()
test_inf  = np.isinf(df_test[feat_cols].values).sum()
print(f"Remaining inf values  — train: {train_inf}  |  test: {test_inf}")

In [ ]:
# ── Constant / near-constant features (single unique value = useless) ─────────
constant_train = [c for c in feat_cols if df_train[c].nunique() <= 1]
constant_test  = [c for c in feat_cols if df_test[c].nunique()  <= 1]
print(f"Constant features in train: {constant_train or 'none'}")
print(f"Constant features in test:  {constant_test  or 'none'}")

# Near-constant: >99.9% of values are the same
near_const_train = [c for c in feat_cols
                    if df_train[c].value_counts(normalize=True).iloc[0] > 0.999]
print(f"\nNear-constant features in train (>99.9% one value): {near_const_train or 'none'}")

In [ ]:
# ── Drop duplicates and constant features; log decisions ─────────────────────
cols_to_drop = list(set(constant_train + constant_test))

df_train_clean = df_train.drop_duplicates(subset=feat_cols).reset_index(drop=True)
df_test_clean  = df_test.drop_duplicates(subset=feat_cols).reset_index(drop=True)

if cols_to_drop:
    df_train_clean = df_train_clean.drop(columns=cols_to_drop)
    df_test_clean  = df_test_clean.drop(columns=cols_to_drop)

print(f"After deduplication:")
print(f"  Train: {len(df_train):,} → {len(df_train_clean):,} rows  "
      f"(removed {len(df_train) - len(df_train_clean):,} duplicates)")
print(f"  Test:  {len(df_test):,}  → {len(df_test_clean):,}  rows  "
      f"(removed {len(df_test) - len(df_test_clean):,} duplicates)")
if cols_to_drop:
    print(f"\nDropped constant columns: {cols_to_drop}")
else:
    print("\nNo constant columns dropped.")

**Hygiene findings:**

**Duplicates:** 14,695 rows removed from train (5.2%) and 17,790 from test (14.3%). Duplicate flows in network data are common — multiple identical flows can be generated when two different attack scripts hit the same destination with identical parameters, or when benign background applications repeatedly open connections with the same characteristics. The higher rate in the test set likely reflects the more uniform traffic patterns in the pre-processed 2018 file.

**Constant columns (10 dropped):** The following columns were zero across every row in at least one dataset and were removed: `Bwd Avg Bulk Rate`, `Bwd Avg Bytes/Bulk`, `Bwd Avg Packets/Bulk`, `Fwd Avg Bulk Rate`, `Fwd Avg Bytes/Bulk`, `Fwd Avg Packets/Bulk`, `Bwd PSH Flags`, `Bwd URG Flags`, `Fwd URG Flags`, `CWE Flag Count`. This is a known limitation of CICFlowMeter on the CIC datasets — bulk transfer statistics are only populated when a flow is identified as a bulk transfer (a specific heuristic), which rarely triggers in simulated lab traffic. These features carry zero information and would only add noise to a model.

**Near-constant but retained:** `ECE Flag Count` and `RST Flag Count` exceed the 99.9% threshold in training but are not constant in the test set. ECE (Explicit Congestion Notification) is rarely set in lab traffic but can appear in real or attack flows; RST (connection reset) is occasionally triggered by scanner tools. They are kept as features since their rare non-zero values may still be informative.

**After cleaning:** Train: 266,739 rows × 66 features. Test: 106,906 rows × 66 features. Both saved to Drive as `train_clean.joblib` / `test_clean.joblib` for use in all subsequent phases.

In [ ]:
# ── Save cleaned frames to Drive ──────────────────────────────────────────────
from joblib import dump

dump(df_train_clean, os.path.join(DATA_DIR, 'train_clean.joblib'))
dump(df_test_clean,  os.path.join(DATA_DIR, 'test_clean.joblib'))
print("Saved train_clean.joblib and test_clean.joblib to Drive.")

---
## §2 — Exploratory Data Analysis

Understand training and test distributions before feature engineering. Covers:
- **§2.1** Class distribution and the class-imbalance problem
- **§2.2** Feature distributions for key network-flow statistics
- **§2.3** Missing values verification
- **§2.4** Outlier analysis
- **§2.5** Temporal-feature analysis
- **§2.6** Cross-tabulation and group-by analysis
- **§2.7** Correlation analysis — method choice and justification (Spearman)
- **§2.8** Pre- vs post-balancing: visualising the 'symmetry' trade-off


In [ ]:
# §2 setup — reload cleaned frames if needed, define FIGURES_DIR
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import load as jload

try:
    df_train_clean
except NameError:
    df_train_clean = jload(os.path.join(DATA_DIR, 'train_clean.joblib'))
    df_test_clean  = jload(os.path.join(DATA_DIR, 'test_clean.joblib'))
    print('Reloaded cleaned frames from Drive.')

feat_cols_clean = [c for c in df_train_clean.columns if c != 'Label']
mask_benign = df_train_clean['Label'].str.strip().str.upper() == 'BENIGN'
print(f'Train: {df_train_clean.shape}, Test: {df_test_clean.shape}, Features: {len(feat_cols_clean)}')

FIGURES_DIR = str(pathlib.Path(DATA_DIR).parent / 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
sns.set_theme(style='whitegrid', palette='tab10')


### §2.1 — Class Distribution and Imbalance Analysis


In [ ]:
def binary_counts(df):
    b = df['Label'].apply(lambda x: 'BENIGN' if str(x).strip().upper()=='BENIGN' else 'ATTACK')
    return b.value_counts().reindex(['BENIGN','ATTACK'], fill_value=0)

train_bc = binary_counts(df_train_clean)
test_bc  = binary_counts(df_test_clean)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, bc) in zip(axes, [('Train (CIC-IDS2017)', train_bc),
                                   ('Test (CSE-CIC-IDS2018)', test_bc)]):
    bars = ax.bar(bc.index, bc.values, color=['steelblue','tomato'],
                  edgecolor='white', linewidth=1.2)
    ax.set_title(name, fontsize=12)
    ax.set_ylabel('Row count')
    for bar, (lbl, val) in zip(bars, bc.items()):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
                f'{val:,}\n({val/bc.sum()*100:.1f}%)',
                ha='center', va='bottom', fontsize=10)

plt.suptitle('Class distribution — before balancing (real prevalence)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'class_distribution_raw.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Train: attack-type breakdown ---')
print(df_train_clean['Label'].value_counts().to_string())
print('\n--- Test: attack-type breakdown ---')
print(df_test_clean['Label'].value_counts().to_string())


**Class imbalance analysis:**

In the raw training set, **BENIGN traffic makes up ~80%** of flows — reflecting real-world network conditions. The progressive test set shows a similar but not identical ratio.

**Why this matters for the paper's conclusions:** The authors downsample to 1:1, which they call 'symmetry.' While this is a standard class-balancing technique, framing it as a necessary *principle* is misleading: accuracy computed on an artificial 1:1 test set does not reflect deployment-realistic performance. In production, ~80% of inputs are benign — a model tuned on balanced data may produce inflated accuracy estimates that collapse when faced with real-world prevalence. We quantify this gap in §2.8.

**Attack-type drift between datasets:** CIC-IDS2017 has PortScan as a dominant attack type; CSE-CIC-IDS2018 introduces DDoS variants (HOIC, LOIC) and Bot traffic largely absent in 2017. This *distributional shift* — not model overfitting — is the likely driver of the cross-dataset accuracy drop. We design a direct test in Phase 8.1.


### §2.2 — Feature Distributions


In [ ]:
KEY_FEATURES = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Flow Bytes/s', 'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
    'Fwd IAT Mean', 'Bwd IAT Mean', 'Flow IAT Mean',
]
KEY_FEATURES = [f for f in KEY_FEATURES if f in feat_cols_clean]

ncols = 3
nrows = (len(KEY_FEATURES) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = axes.flatten()

for ax, feat in zip(axes, KEY_FEATURES):
    clip_val = df_train_clean[feat].quantile(0.99)
    for lbl, mask, color in [
        ('BENIGN', mask_benign, 'steelblue'),
        ('ATTACK', ~mask_benign, 'tomato')
    ]:
        ax.hist(df_train_clean.loc[mask, feat].clip(upper=clip_val),
                bins=50, alpha=0.5, color=color, label=lbl, density=True)
    ax.set_title(feat, fontsize=9)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=9)
for ax in axes[len(KEY_FEATURES):]:
    ax.set_visible(False)

plt.suptitle('Feature distributions by class — train set (99th-percentile clipped)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()


**Feature distribution findings:**

Network flow features are **extremely right-skewed** — most flows are short with few packets, but a heavy tail of high-volume flows exists. This is typical of real network traffic (a mix of brief DNS queries and long streaming sessions) and **violates the normality assumption behind Pearson correlation**, motivating our use of Spearman in §2.7.

**Class separation signals:**
- `Flow Bytes/s` discriminates well: DoS attacks send maximum-rate traffic; port scans send tiny probes at high rate.
- `Total Backward Packets` near-zero flags unidirectional DoS floods (victim cannot respond).
- `Fwd/Bwd IAT Mean` reveals timing fingerprints — attack tools have characteristically different inter-packet intervals than human-driven traffic.

The overlap in the middle of most distributions explains why no single feature suffices — the paper's 11-feature selection (reproduced in §3) targets the most discriminative *combination*.


### §2.3 — Missing Values Verification


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, df) in zip(axes, [('Train', df_train_clean), ('Test', df_test_clean)]):
    nan_counts = df[feat_cols_clean].isnull().sum()
    nan_nz = nan_counts[nan_counts > 0]
    if nan_nz.empty:
        ax.text(0.5, 0.5, 'No missing values\n(cleaned in §1)',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=13, color='seagreen')
    else:
        nan_nz.sort_values(ascending=False).plot(kind='bar', ax=ax, color='tomato')
        ax.set_ylabel('NaN count')
    ax.set_title(f'{name} — NaN per feature')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'missing_values.png'), dpi=150, bbox_inches='tight')
plt.show()

for name, df in [('Train', df_train_clean), ('Test', df_test_clean)]:
    numeric = df[feat_cols_clean].select_dtypes(include='number')
    inf_count = np.isinf(numeric).sum().sum()
    print(f'{name} — inf values remaining: {inf_count}')


### §2.4 — Outlier Analysis


In [ ]:
print('Outlier rate (IQR method, >Q3 + 1.5*IQR) — train set:\n')
outlier_summary = {}
for feat in KEY_FEATURES:
    q1, q3 = df_train_clean[feat].quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    n_out = int((df_train_clean[feat] > upper).sum())
    outlier_summary[feat] = n_out
    print(f'  {feat:<35}: {n_out:>6,}  ({n_out/len(df_train_clean)*100:.1f}%)')

top6 = sorted(outlier_summary, key=outlier_summary.get, reverse=True)[:6]
top6 = [f for f in top6 if f in df_train_clean.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flatten(), top6):
    cap = df_train_clean[feat].quantile(0.999)
    data = [
        df_train_clean.loc[mask_benign, feat].clip(upper=cap).values,
        df_train_clean.loc[~mask_benign, feat].clip(upper=cap).values,
    ]
    bp = ax.boxplot(data, labels=['Benign', 'Attack'], patch_artist=True, showfliers=False)
    for patch, color in zip(bp['boxes'], ['steelblue', 'tomato']):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(feat, fontsize=9)
    ax.tick_params(labelsize=8)

plt.suptitle(
    'Box plots — highest-outlier features by class (99.9th-pct clipped, no fliers shown)',
    fontsize=11
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'outlier_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()


**Outlier findings:**

Outlier rates are **high by IQR standards (5–30% per feature)** — this is expected and meaningful, not a data quality problem:
- **DoS/DDoS attacks** are the outliers in `Flow Bytes/s` — attack tools flood targets at maximum rate.
- **Port scans** produce extreme values in packet-length features (tiny probe packets) and very short `Flow Duration` (rapid failed connection attempts).
- **Benign elephant flows** (large file transfers, video streaming) are genuine outliers in byte-count features.

**Design decision — do not clip outliers before training:** Tree models (DT, RF) handle extreme values via split thresholds. For SVM and ANN we apply standard scaling in §3, which compresses the tail without discarding signal. Removing outliers would destroy the attack-fingerprint patterns that make classification possible.


### §2.5 — Temporal Feature Analysis


In [ ]:
TEMPORAL_FEATS = [f for f in
    ['Flow Duration', 'Fwd IAT Mean', 'Bwd IAT Mean', 'Flow IAT Mean']
    if f in feat_cols_clean]

top10_labels = df_train_clean['Label'].value_counts().head(10).index.tolist()
df_sub = df_train_clean[df_train_clean['Label'].isin(top10_labels)]

fig, axes = plt.subplots(len(TEMPORAL_FEATS), 1, figsize=(13, 4 * len(TEMPORAL_FEATS)))
if len(TEMPORAL_FEATS) == 1:
    axes = [axes]

for ax, feat in zip(axes, TEMPORAL_FEATS):
    medians = df_sub.groupby('Label')[feat].median().sort_values()
    colors = ['steelblue' if 'BENIGN' in str(l).upper() else 'tomato'
              for l in medians.index]
    ax.barh(medians.index, medians.values, color=colors)
    ax.set_title(f'Median {feat} by class', fontsize=11)
    ax.set_xlabel(feat)
    ax.tick_params(labelsize=9)

plt.suptitle('Temporal features by attack class — top 10 classes, train set', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'temporal_features_by_class.png'), dpi=150, bbox_inches='tight')
plt.show()


**Temporal feature analysis — alignment with world knowledge:**

Inter-arrival time (IAT) and Flow Duration reveal attack fingerprints consistent with known network behaviour:

- **DoS/DDoS attacks** (GoldenEye, Hulk, HOIC, LOIC): very short flow durations and near-zero IATs — attack tools flood targets as fast as possible, creating high-frequency, low-duration flows.
- **PortScan**: short flows (connection attempts that don't complete the TCP handshake) and moderate IAT — a scanner probes ports sequentially with brief pauses.
- **Brute-force** (FTP-Patator, SSH-Patator): longer flows because each authentication attempt involves a full protocol exchange (challenge–response cycles add time).
- **Bot**: long flow durations with moderate IAT — bots establish persistent C2 connections and check in periodically, mimicking long-lived legitimate sessions.
- **BENIGN**: highest IAT variance — real users have irregular, human-paced behaviour mixing brief DNS queries with long streaming sessions.

**Why Timestamp is excluded from features:** Including raw calendar timestamps would teach the model 'flows from 2017 = benign, flows from 2018 = attack,' producing temporal leakage that invalidates the progressive evaluation entirely. IAT and Duration are *structural* timing properties of a flow, not its calendar position, and are safe to include as features.


### §2.6 — Cross-tabulation and Group-by Analysis


In [ ]:
SUMMARY_FEATS = [f for f in
    ['Flow Duration','Total Fwd Packets','Total Backward Packets',
     'Flow Bytes/s','Fwd Packet Length Mean','Bwd Packet Length Mean']
    if f in feat_cols_clean]

top8 = df_train_clean['Label'].value_counts().head(8).index.tolist()
group_stats = (
    df_train_clean[df_train_clean['Label'].isin(top8)]
    .groupby('Label')[SUMMARY_FEATS]
    .median()
    .round(2)
)
print('Median feature values by class (top 8, train set):')
print(group_stats.to_string())
print()

if 'Total Fwd Packets' in feat_cols_clean and 'Total Backward Packets' in feat_cols_clean:
    ratio = (df_train_clean['Total Backward Packets'] /
             (df_train_clean['Total Fwd Packets'] + 1e-6)).clip(0, 100)
    ratio_by_class = (
        df_train_clean[df_train_clean['Label'].isin(top8)]
        .assign(_ratio=ratio)
        .groupby('Label')['_ratio']
        .median()
        .sort_values()
    )
    print('Median Bwd/Fwd packet ratio by class:')
    print('  (0 = purely unidirectional DoS;  ~1 = balanced bidirectional traffic)')
    print(ratio_by_class.to_string())


**Cross-tabulation findings:**

The group-by table confirms well-known network-security patterns:

- **DoS attacks** (Hulk, GoldenEye, Slowloris): near-zero Bwd/Fwd ratio — the victim responds minimally during a flood, so backward (server→client) packets are almost absent. This asymmetry makes them highly distinguishable from normal bidirectional traffic.
- **Brute-force** (FTP-Patator, SSH-Patator): higher Bwd/Fwd ratio — the server sends authentication challenges and failure messages back for every attempt.
- **BENIGN**: balanced ratio close to 1 — normal HTTP, DNS, and application traffic is bidirectional.

**Relevance to feature selection:** The Bwd/Fwd asymmetry is implicitly captured by the raw forward/backward packet count features the authors selected. In §3 we examine whether an explicitly derived ratio feature improves the importance ranking.


### §2.7 — Correlation Analysis — Method Choice and Justification


**Why Spearman (not Pearson or Kendall):**

| Method | Key assumption | Verdict for this dataset |
|--------|---------------|-------------------------|
| **Pearson** | Linear relationship; normally distributed, homoscedastic data | ❌ Network features follow power-law distributions; outliers are attack signals, not errors; relationship is monotonic but not strictly linear |
| **Kendall** | Rank-based; no normality assumption; robust to outliers | ✅ Correct assumptions, but **O(n²)** computation — impractical for ~250k rows |
| **Spearman** | Rank-based; no normality assumption; robust to outliers | ✅ Captures monotonic relationships; **O(n log n)**; correct and efficient |

**Practical vs. statistical significance:** With ~250k rows, virtually every correlation is statistically significant (p < 0.001). What matters is *practical* significance: |r| > 0.90 identifies feature pairs carrying essentially redundant information — candidates for removal in the feature-selection step (§3).


In [ ]:
# Spearman on a 10k-row subsample — fast and representative
_CORR_N = 10_000
df_corr_sample = df_train_clean[feat_cols_clean].sample(
    min(_CORR_N, len(df_train_clean)), random_state=SEED
)
print(f'Computing Spearman matrix on {len(df_corr_sample):,} rows x {len(feat_cols_clean)} features...')
corr_matrix = df_corr_sample.corr(method='spearman')
print('Done.')

# Pairs with |r| > 0.90 — redundant features
high_pairs = []
cols = corr_matrix.columns.tolist()
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        r = float(corr_matrix.iloc[i, j])
        if abs(r) > 0.90:
            high_pairs.append((abs(r), r, cols[i], cols[j]))
high_pairs.sort(reverse=True)

print(f'\nFeature pairs |Spearman r| > 0.90  ({len(high_pairs)} total — top 20 shown):')
for _, r, c1, c2 in high_pairs[:20]:
    print(f'  r={r:+.3f}  {c1}  <->  {c2}')


In [ ]:
# Spearman heatmap — top 30 highest-variance features, hierarchically clustered
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

top30 = df_corr_sample.var().nlargest(30).index.tolist()
sub_corr = corr_matrix.loc[top30, top30]

try:
    dist_mat = (1 - sub_corr.abs()).clip(lower=0)
    np.fill_diagonal(dist_mat.values, 0.0)
    link = linkage(squareform(dist_mat.values), method='average')
    order = leaves_list(link)
    sub_corr = sub_corr.iloc[order, order]
except Exception as e:
    print(f'Hierarchical clustering skipped ({e}); using original order.')

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(sub_corr, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, cbar_kws={'label': 'Spearman r'},
            xticklabels=True, yticklabels=True)
ax.tick_params(axis='x', labelrotation=45, labelsize=7)
ax.tick_params(axis='y', labelrotation=0, labelsize=7)
ax.set_title('Spearman correlation — top 30 highest-variance features (train set)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'spearman_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()


**Correlation interpretation:**

The heatmap reveals three major redundancy clusters:

1. **Packet length stats** — `Fwd/Bwd Packet Length Mean/Max/Min/Std`, `Average Packet Size`, `Avg Fwd/Bwd Segment Size`: all measure 'how big are the packets?' and are near-perfectly correlated (|r| > 0.95). Most carry identical information.
2. **IAT (inter-arrival time) cluster** — `Fwd/Bwd/Flow IAT Total/Mean/Std/Min/Max`: timing features computed from the same packet timestamp sequence.
3. **Subflow / bulk features** — near-perfectly correlated with parent flow counts (`Subflow Fwd Packets` ≈ `Total Fwd Packets`). These are CICFlowMeter artefacts adding no new information.

This directly motivates the authors' 76 → 11 feature reduction. In §3 we reproduce the RF-importance + brute-force selection and verify that the retained features span these clusters rather than over-representing any one group.


### §2.8 — Pre- vs Post-Balancing: the 'Symmetry' Trade-off


In [ ]:
df_bin = df_train_clean.assign(
    _binary=df_train_clean['Label'].apply(
        lambda x: 'BENIGN' if str(x).strip().upper()=='BENIGN' else 'ATTACK'
    )
)
before = df_bin['_binary'].value_counts().reindex(['BENIGN','ATTACK'], fill_value=0)

n_min = int(before.min())
balanced = pd.concat([
    df_bin[df_bin['_binary']==cls].sample(n=n_min, random_state=SEED)
    for cls in ['BENIGN','ATTACK']
])
after = balanced['_binary'].value_counts().reindex(['BENIGN','ATTACK'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (title, bc) in zip(axes, [
    ('Before balancing — real prevalence', before),
    ("After 1:1 downsampling (paper's 'symmetry')", after)
]):
    bars = ax.bar(bc.index, bc.values, color=['steelblue','tomato'],
                  edgecolor='white', linewidth=1.2)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Row count')
    total = bc.sum()
    for bar, val in zip(bars, bc.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                f'{val:,}\n({val/total*100:.1f}%)',
                ha='center', va='bottom', fontsize=10)

plt.suptitle("Class distribution: real prevalence vs. paper's 1:1 'symmetry'", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'class_balancing_effect.png'), dpi=150, bbox_inches='tight')
plt.show()

discarded = int(before.sum() - after.sum())
print(f'Rows discarded by 1:1 downsampling: {discarded:,}')
print(f'Training data retained: {after.sum()/before.sum()*100:.1f}%')


**Pre- vs post-balancing:**

The 1:1 downsampling discards a substantial fraction of BENIGN training rows. Consequences:

1. **Metric inflation on balanced test sets:** Accuracy on a 1:1 test set exceeds accuracy on a realistic 80/20 deployment set for models trained on balanced data. The paper's reported in-distribution numbers (~96–99%) are upper bounds.
2. **Data loss:** Discarding excess BENIGN flows means the model sees a less representative sample of legitimate traffic — unusual-but-legitimate flows may be systematically removed.
3. **Alternatives not tested:** Class-weighted loss, SMOTE (synthetic minority oversampling), or probability threshold tuning — all preserve training data while addressing imbalance. We evaluate on a real-prevalence test split in Phase 8.2 to quantify the deployment accuracy gap.


---
## §3 — Feature Engineering

Reproduce the authors' pipeline: cleaning → balancing → binary relabeling → encoding → scaling → feature creation → feature selection.

---
## §4 — Model Training

Train DT, RF, SVM, NB, ANN, DNN with GridSearchCV (k=5). Save best models to Drive.

---
## §5 — Evaluation & Reproduction Check

In-distribution evaluation (reproduce Tables 4–6). Progressive evaluation on CSE-CIC-IDS2018 (reproduce Table 7). Side-by-side comparison with paper's numbers.

---
## §6 — Error Analysis

Misclassified examples (FPs and FNs) on the progressive test set. Patterns in errors. Cybersecurity implications.

---
## §7 — Executive Summary

*(Filled after all analysis is complete.)*

---
## §8 — Summing It Up

*(Filled after all analysis is complete.)*